In [ ]:
import pandas as pd
import numpy as np

input_path = "/mnt/data/NIFTY50_CLEANED.csv"
output_path = "/mnt/data/NIFTY50_TARGETED.csv"

df = pd.read_csv(input_path)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Define the next trading day's close and return.
df["Next_Close"] = df["Close"].shift(-1)
df["Next_Day_Return"] = (df["Next_Close"] - df["Close"]) / df["Close"]

# Binary target:
# 1 = UP, 0 = DOWN.
# Exact equal closes are neither UP nor DOWN, so they are excluded.
df["Target"] = np.where(
    df["Next_Close"].isna(),
    np.nan,
    np.where(df["Next_Close"] > df["Close"], 1, 0)
)

equal_mask = (df["Next_Close"] == df["Close"]) & df["Next_Close"].notna()
equal_rows = int(equal_mask.sum())

# Remove the final row (no next-day target) and the exact-equality row.
targeted = df.loc[df["Next_Close"].notna() & ~equal_mask].copy()

# Keep Next_Day_Return for auditing/analysis, but it must NOT be used as an ML feature.
targeted = targeted[
    ["Date", "Open", "High", "Low", "Close", "Next_Day_Return", "Target"]
]

targeted["Target"] = targeted["Target"].astype(int)

targeted.to_csv(output_path, index=False)

up = int((targeted["Target"] == 1).sum())
down = int((targeted["Target"] == 0).sum())
total = len(targeted)

print(f"Created: {output_path}")
print(f"Original rows: {len(df)}")
print(f"Rows after target creation: {total}")
print(f"UP: {up} ({up/total*100:.2f}%)")
print(f"DOWN: {down} ({down/total*100:.2f}%)")
print(f"Exact equal-close row excluded: {equal_rows}")
print("\nFirst 5 rows:")
print(targeted.head().to_string(index=False))
